In [2]:
import sys
# Use an absolute path or relative path to the directory
sys.path.append("scripts")

import numpy as np
import pdb_voxelizier
import cnn_mlp_encoder
import jw_quantum_mapper
from scipy.optimize import minimize
from scipy.spatial.distance import squareform
from scipy.linalg import eigh
import sympy
import openfermion as op
import pyvista as pv
import torch
import cirq

In [ ]:
import os
import torch
import matplotlib.pyplot as plt
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import warnings
!pip install biopython
from Bio.PDB import PDBParser

In [ ]:
num_sites = 4
tensor = pdb_voxelizier.pdb_to_tensor('proteins/1ENH.pdb', grid_size = 64)

#coefficients = cnn_mlp_encoder.get_hamiltonian(tensor, num_qubits=num_sites)
#qubit_instructions = jw_quantum_mapper.apply_jw(coefficients,num_sites=num_sites)

# qubit_instructions

Get feature maps for first convolutional layer 

In [ ]:
def visualize_feature_maps(layer, image, slice_index=None):
    #so gradients are not calculated when running 
    with torch.no_grad():
        # Remove batch dimension, so batchsize (batchsize, channels, dxhxw)
        maps = layer(image).squeeze(0).cpu()
    # calculates subplot rows 
    # len: gives how many items are in a tensor of feature maps 
    rows = (len(maps) + 3) // 4
    # creates figure; widthxheightxrows, 16x4 just general choice 
    plt.figure(figsize=(16, 4 * rows))
    # loop that goes through every feature map 
    # enumerate: adds an index so converts list into index
    for i, fmap in enumerate(maps):
        # chooses the slice
        z = slice_index if slice_index is not None else fmap.shape[0] // 2
        # gets the 2d slice 
        slice_img = fmap[z].numpy()
        # normalizes the values 
        slice_img = (slice_img - slice_img.min()) / (slice_img.max() - slice_img.min() + 1e-8)
        # create subplot and display 
        plt.subplot(rows, 4, i + 1)
        plt.imshow(slice_img, cmap="viridis")
        plt.title(f"Map {i+1}")
        plt.axis("off")

    plt.tight_layout()
    plt.show()

In [ ]:
model = ProteinPhysicsEncoder(num_sites=4)
# converts data into pytorch tensor 
input_tensor = torch.FloatTensor(tensor)
# try changing slice to see different sections of 3d feature maps  
visualize_feature_maps(model.conv1, input_tensor, slice_index=24)


get feature maps for second convolutional layer 

In [ ]:
# this is the forward pass function as need to apply first convolutional layer and the operations inorder
x = model.pool1(F.relu(model.conv1(input_tensor)))
# try changing slice to see different sections of 3d feature maps  
visualize_feature_maps( model.conv2, x, slice_index = 8 )

In [ ]:
def visualize_filters(layer):
# get learned filter weights 
# detach = removes weights from computation graph, so it doesn't compute gradients only weights
  filters = layer.weight.detach().cpu()
# shape (out channels, in channels, depth, height, width), so only takes out channels needed
  num_filters = filters.shape[0]
# creates figure, just general size 12x8
  plt.figure(figsize = (12, 8))
# loop to go through every filter 
  for i in range(num_filters): 
    # get one filter, so first input channel 
    filter = filters[i, 0]
    # show middle slide of 3d filter, 
    middle_slice = filter[1]
    # create grid of plots 
    plt.subplot(4,4, i + 1)
    # convert numbers to pixels and use grayscale so positive weights = lighter vs negative = darker
    plt.imshow(middle_slice, cmap = "viridis")
    plt.axis("off")
    plt.title(f"Map {i+1}")

plt.show()


In [ ]:
visualize_filters(model.conv1)


In [ ]:
visualize_filters(model.conv2)
